# 04 — Modeling & Tuning

Two model families (Linear/Ridge Regression + Decision Tree or AdaBoost), compared under a standard random split AND a spatially-grouped split (GroupKFold by neighbourhood). This comparison is the project's core contribution — quantifying spatial leakage in the published benchmarks this project's literature review is based on.

In [1]:
import sys
sys.path.append('../src')
from utils import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

## 1. Load final feature set


In [2]:
# ── 1. Load final feature set ─────────────────────────────────────
df = pd.read_csv('../data/processed/airbnb_features.csv')
print(df.shape)

non_feature_cols = ['price', 'price_log', 'neighbourhood', 'neighbourhood_group',
                     'room_type', 'geo_cluster_raw', 'host_scale_raw']
feature_cols = [c for c in df.columns if c not in non_feature_cols]

X = df[feature_cols]
y = df['price_log']
groups = df['neighbourhood']

print(f"Model features ({len(feature_cols)}): {feature_cols}")
print(f"Unique neighbourhoods (grouping key): {groups.nunique()}")

(45899, 19)
Model features (12): ['boro_Brooklyn', 'room_Shared room', 'room_Private room', 'availability_365_scaled', 'never_reviewed', 'boro_Staten Island', 'boro_Queens', 'boro_Manhattan', 'host_small', 'host_individual', 'geo_3', 'geo_1']
Unique neighbourhoods (grouping key): 219


## 2. Split A — random train/val/test
Standard sklearn train_test_split, replicating prior published methodology.

In [3]:
# ── 2. Split A — random train/test ────────────────────────────────
from sklearn.model_selection import train_test_split

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Split A (random) — train: {X_train_a.shape}, test: {X_test_a.shape}")

Split A (random) — train: (36719, 12), test: (9180, 12)


## 3. Split B — spatially grouped
GroupKFold or manual split holding out entire neighbourhoods for val/test.

In [4]:
# ── 3. Split B — spatially grouped (holds out entire neighbourhoods) ──
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_b, X_test_b = X.iloc[train_idx], X.iloc[test_idx]
y_train_b, y_test_b = y.iloc[train_idx], y.iloc[test_idx]
groups_train_b = groups.iloc[train_idx]

train_n = set(groups.iloc[train_idx])
test_n = set(groups.iloc[test_idx])
print(f"Split B (spatial) — train: {X_train_b.shape}, test: {X_test_b.shape}")
print(f"Neighbourhoods — train: {len(train_n)}, test: {len(test_n)}, overlap: {len(train_n & test_n)}")

Split B (spatial) — train: (38664, 12), test: (7235, 12)
Neighbourhoods — train: 175, test: 44, overlap: 0


## 4. Train baseline: Linear/Ridge Regression
Fit on both splits.

In [5]:
# ── 4. Ridge Regression — Split A (random), grid search + CV ─────
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

ridge_param_grid = {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}

ridge_grid_a = GridSearchCV(Ridge(), ridge_param_grid, cv=5,
                             scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_grid_a.fit(X_train_a, y_train_a)

results_a = pd.DataFrame(ridge_grid_a.cv_results_)[['param_alpha', 'mean_test_score', 'std_test_score']]
results_a['mean_test_score'] = -results_a['mean_test_score']
print("Ridge grid search (Split A):")
print(results_a.sort_values('mean_test_score'))
print(f"Best alpha: {ridge_grid_a.best_params_}")

ridge_best_a = ridge_grid_a.best_estimator_
pred_ridge_a = ridge_best_a.predict(X_test_a)
metrics_ridge_a = regression_report(y_test_a, pred_ridge_a, label="Ridge (random split, log scale)")

Ridge grid search (Split A):
   param_alpha  mean_test_score  std_test_score
2         1.00         0.380870        0.003103
1         0.10         0.380870        0.003106
0         0.01         0.380870        0.003106
3        10.00         0.380879        0.003072
4       100.00         0.381579        0.002919
Best alpha: {'alpha': 1.0}
Ridge (random split, log scale) RMSE: 0.3800 | MAE: 0.2981 | R2: 0.5563


## 5. Train model 2: Decision Tree / AdaBoost
Fit on both splits.

In [6]:
# ── 5. AdaBoost (CART base) — Split A (random), grid search + CV ─
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

# NOTE: sklearn >=1.2 uses `estimator=`; older versions use `base_estimator=`
# and `base_estimator__max_depth` instead of `estimator__max_depth`.
ada_param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 1.0],
    'estimator__max_depth': [2, 3, 4],
}

ada_base = AdaBoostRegressor(estimator=DecisionTreeRegressor(random_state=42), random_state=42)
ada_grid_a = GridSearchCV(ada_base, ada_param_grid, cv=5,
                           scoring='neg_root_mean_squared_error', n_jobs=-1)
ada_grid_a.fit(X_train_a, y_train_a)

results_ada_a = pd.DataFrame(ada_grid_a.cv_results_)[
    ['param_n_estimators', 'param_learning_rate', 'param_estimator__max_depth', 'mean_test_score']
]
results_ada_a['mean_test_score'] = -results_ada_a['mean_test_score']
print("AdaBoost grid search (Split A) — top 5:")
print(results_ada_a.sort_values('mean_test_score').head())
print(f"Best params: {ada_grid_a.best_params_}")

ada_best_a = ada_grid_a.best_estimator_
pred_ada_a = ada_best_a.predict(X_test_a)
metrics_ada_a = regression_report(y_test_a, pred_ada_a, label="AdaBoost (random split, log scale)")

AdaBoost grid search (Split A) — top 5:
    param_n_estimators  param_learning_rate  param_estimator__max_depth  \
19                 100                 0.01                           4   
18                  50                 0.01                           4   
20                 200                 0.01                           4   
21                  50                 0.10                           4   
12                  50                 0.10                           3   

    mean_test_score  
19         0.381271  
18         0.381354  
20         0.381448  
21         0.382792  
12         0.386580  
Best params: {'estimator__max_depth': 4, 'learning_rate': 0.01, 'n_estimators': 100}
AdaBoost (random split, log scale) RMSE: 0.3828 | MAE: 0.3004 | R2: 0.5500


## 6. Hyperparameter tuning
GridSearchCV — report search space and full results table, not just the best score.

In [7]:
# ── 6. Same two models — Split B (spatial), group-aware CV ───────
group_kfold = GroupKFold(n_splits=5)

ridge_grid_b = GridSearchCV(Ridge(), ridge_param_grid, cv=group_kfold,
                             scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_grid_b.fit(X_train_b, y_train_b, groups=groups_train_b)
ridge_best_b = ridge_grid_b.best_estimator_
pred_ridge_b = ridge_best_b.predict(X_test_b)
metrics_ridge_b = regression_report(y_test_b, pred_ridge_b, label="Ridge (spatial split, log scale)")
print(f"Best alpha (spatial): {ridge_grid_b.best_params_}")

ada_base_b = AdaBoostRegressor(estimator=DecisionTreeRegressor(random_state=42), random_state=42)
ada_grid_b = GridSearchCV(ada_base_b, ada_param_grid, cv=group_kfold,
                           scoring='neg_root_mean_squared_error', n_jobs=-1)
ada_grid_b.fit(X_train_b, y_train_b, groups=groups_train_b)
ada_best_b = ada_grid_b.best_estimator_
pred_ada_b = ada_best_b.predict(X_test_b)
metrics_ada_b = regression_report(y_test_b, pred_ada_b, label="AdaBoost (spatial split, log scale)")
print(f"Best params (spatial): {ada_grid_b.best_params_}")

Ridge (spatial split, log scale) RMSE: 0.3844 | MAE: 0.2991 | R2: 0.5786
Best alpha (spatial): {'alpha': 1.0}


AdaBoost (spatial split, log scale) RMSE: 0.3866 | MAE: 0.3022 | R2: 0.5738
Best params (spatial): {'estimator__max_depth': 4, 'learning_rate': 0.01, 'n_estimators': 100}


## 7. Compare random-split vs. grouped-split performance
This is your key result — quantify the gap.

In [8]:
# ── 7. Comparison summary — this is your core result ─────────────
comparison = pd.DataFrame([
    {'model': 'Ridge', 'split': 'random', **metrics_ridge_a},
    {'model': 'Ridge', 'split': 'spatial', **metrics_ridge_b},
    {'model': 'AdaBoost', 'split': 'random', **metrics_ada_a},
    {'model': 'AdaBoost', 'split': 'spatial', **metrics_ada_b},
])
print(comparison)
comparison.to_csv('../data/processed/model_comparison_summary.csv', index=False)

      model    split      rmse       mae        r2
0     Ridge   random  0.380050  0.298128  0.556333
1     Ridge  spatial  0.384433  0.299064  0.578552
2  AdaBoost   random  0.382757  0.300406  0.549990
3  AdaBoost  spatial  0.386581  0.302237  0.573830


## 8. Save best models
Pickle/joblib to `../models/`.

In [9]:
# ── 8. Save models ─────────────────────────────────────────────────
import joblib

joblib.dump(ridge_best_a, '../models/ridge_random_split.pkl')
joblib.dump(ridge_best_b, '../models/ridge_spatial_split.pkl')
joblib.dump(ada_best_a, '../models/adaboost_random_split.pkl')
joblib.dump(ada_best_b, '../models/adaboost_spatial_split.pkl')
print("Models saved to ../models/")

Models saved to ../models/
